In [9]:
import pandas as pd
import glob
import os
import re

# 1. Leer todos los archivos JSONL de las categorías (excluyendo el consolidado masivo)
patron_archivos = "../datalake_crudo/chilepropiedades/chilepropiedades_*.jsonl"
archivos = [f for f in glob.glob(patron_archivos) if "masivo" not in f]

lista_df = [pd.read_json(archivo, lines=True) for archivo in archivos]
df = pd.concat(lista_df, ignore_index=True)

# Eliminamos duplicados por URL
df = df.drop_duplicates(subset=['url']).reset_index(drop=True)

# 2. Extraer comuna y tipo_propiedad directamente desde la estructura de la URL
# Ejemplo: https://chilepropiedades.cl/ver-publicacion/venta/la-florida/casa/...
extraido = df['url'].str.extract(r'ver-publicacion/[^/]+/([^/]+)/([^/]+)/')
df['comuna'] = extraido[0]
df['tipo_propiedad'] = extraido[1]

# 3. Limpiar el texto automático de ChilePropiedades (tanto la versión larga como la corta)
patron_basura = r"^.*?aprobación de crédito\.\s*"
df['descripcion_limpia'] = (
    df['descripcion_completa']
    .fillna("")
    .str.replace(patron_basura, "", regex=True)
    .str.strip()
)

# 4. Filtrar solo publicaciones que tengan una descripción real escrita por el vendedor (>= 30 caracteres)
df_limpio = df[df['descripcion_limpia'].str.len() >= 30].copy().reset_index(drop=True)

print(f"Total publicaciones únicas: {len(df)}")
print(f"Publicaciones sin descripción (solo texto legal): {len(df) - len(df_limpio)}")
print(f"Publicaciones con descripción real listas para NLP: {len(df_limpio)}")

# Mostrar vista previa de las columnas clave
df_limpio[['id_propiedad', 'comuna', 'tipo_propiedad', 'titulo', 'descripcion_limpia']].head(5)

Total publicaciones únicas: 19700
Publicaciones sin descripción (solo texto legal): 18206
Publicaciones con descripción real listas para NLP: 1494


,id_propiedad,comuna,tipo_propiedad,titulo,descripcion_limpia
0,CHP-bod-0-1,providencia,bodega,Bodega en Venta en los Leones,BODEGA EN VENTA – PROVIDENCIA | SECTOR LOS LEO...
1,CHP-bod-0-4,estacion-central,bodega,"Bodega en venta en Buzo Sobenes, Estación Cent...","Galpón muy bien ubicado, excelente conectivida..."
2,CHP-bod-0-5,macul,bodega,"Bodega en venta en Exequiel Fernández 2330, Macul","Propiedad Comercial – Industrial , 1500 m2 , t..."
3,CHP-bod-0-6,el-bosque,bodega,"Bodega en venta en Larga, El Bosque | UF 5.380",Venta de Bodega en Gran Avenida Paradero 35 Co...
4,CHP-bod-2-3,las-condes,bodega,"Bodega en venta en Avenida Las Condes, Las Con...",Vende bodega de 14 m² muy luminosa y funcional...


In [10]:
from transformers import pipeline
import torch
from tqdm.auto import tqdm

# Detectar GPU o CPU
dispositivo = 0 if torch.cuda.is_available() else -1
print(f"Dispositivo de ejecución: {'GPU' if dispositivo == 0 else 'CPU'}")

# Cargamos modelo Zero-Shot entrenado en español
clasificador = pipeline(
    "zero-shot-classification",
    model="Recognai/zeroshot_selectra_medium",
    device=dispositivo
)

# Etiquetas basadas en las descripciones reales de tu JSONL y lo pedido por tu compañero
etiquetas = [
    "cercano al metro",
    "estacionamiento",
    "terraza o patio",
    "buena conectividad",
    "piscina o quincho",
    "seguridad o condominio cerrado",
    "bodega",
    "ampliación o remodelada"
]

# Tomamos una muestra reproducible de 100 publicaciones con descripción real
muestra_nlp = df_limpio.sample(n=100, random_state=42).copy().reset_index(drop=True)

resultados_etiquetas = []
resultados_scores = []

print("Procesando descripciones con Hugging Face...")
for texto in tqdm(muestra_nlp['descripcion_limpia'], total=len(muestra_nlp)):
    # Recortamos a los primeros 600 caracteres para optimizar velocidad y límite de tokens
    pred = clasificador(
        texto[:600],
        candidate_labels=etiquetas,
        multi_label=True,
        hypothesis_template="Esta propiedad cuenta con {}."
    )
    
    # Guardamos las características que superen el umbral de confianza (0.65)
    detectadas = [label for label, score in zip(pred['labels'], pred['scores']) if score >= 0.65]
    scores_dict = {label: round(score, 3) for label, score in zip(pred['labels'], pred['scores'])}
    
    resultados_etiquetas.append(detectadas)
    resultados_scores.append(scores_dict)

muestra_nlp['caracteristicas_nlp'] = resultados_etiquetas
muestra_nlp['scores_nlp'] = resultados_scores

# Mostramos los primeros 5 resultados enriquecidos
for idx, row in muestra_nlp.head(5).iterrows():
    print(f"\n[{row['id_propiedad']}] ({row['comuna'].upper()} - {row['tipo_propiedad']})")
    print(f"Texto: {row['descripcion_limpia'][:180]}...")
    print(f"Etiquetas extraídas por NLP: {row['caracteristicas_nlp']}")

c:\Users\extac\OneDrive\Escritorio\Proyecto_inmobiliario\mi_entorno\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo de ejecución: CPU


c:\Users\extac\OneDrive\Escritorio\Proyecto_inmobiliario\mi_entorno\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\extac\.cache\huggingface\hub\models--Recognai--zeroshot_selectra_medium. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\extac\OneDrive\Escritorio\Proyecto_inmobiliario\mi_entorno

Procesando descripciones con Hugging Face...


100%|██████████| 100/100 [01:07<00:00,  1.49it/s]


[CHP-dep-804-2] (SANTIAGO - departamento)
Texto: 📍 Santo Domingo 504, frente al Museo y a ½ cuadra del Metro ✅ 2 dormitorios | 2 baños ✅ Recién remodelado ✅ Opción 2 estacionamientos (400 UF c/u)...
Etiquetas extraídas por NLP: ['cercano al metro', 'ampliación o remodelada', 'buena conectividad', 'estacionamiento', 'terraza o patio']

[CHP-dep-754-3] (NUNOA - departamento)
Texto: Espectacular departamento, como nuevo Full diseño, cuentacon Cava de vinos, Orientación Norte 1 Estacionamiento Hall de entrada, amplio living y comedor con salida a ricaterraza Or...
Etiquetas extraídas por NLP: ['buena conectividad', 'bodega', 'ampliación o remodelada', 'estacionamiento', 'cercano al metro', 'terraza o patio']

[CHP-par-83-8] (CURACAVI - parcela)
Texto: PARCELA CON JARDÍN FORMADO Y CASA CON FINAS TERMINACIONES Parcela en Curacavi de 17.000 m2, condominio consolidado, ingreso controlado, casa de 260 m2 tipo Metalcom, amplio jardín ...
Etiquetas extraídas por NLP: ['terraza o patio', 'bodega'

In [11]:
import os

# 1. Crear carpeta de salida si no existe y exportar resultados
os.makedirs("../datalake_procesado/nlp", exist_ok=True)

# Seleccionamos columnas limpias para exportar a JSONL y CSV
columnas_exportar = [
    'id_propiedad', 'url', 'comuna', 'tipo_propiedad',
    'precios', 'caracteristicas', 'descripcion_limpia',
    'caracteristicas_nlp', 'scores_nlp'
]

ruta_salida_jsonl = "../datalake_procesado/nlp/chilepropiedades_nlp_enriquecido.jsonl"
muestra_nlp[columnas_exportar].to_json(ruta_salida_jsonl, orient="records", lines=True, force_ascii=False)
print(f"Archivo de resultados guardado exitosamente en: {ruta_salida_jsonl}")

# 2. Reporte de Frecuencia de Características Detectadas
conteo_etiquetas = muestra_nlp['caracteristicas_nlp'].explode().value_counts().reset_index()
conteo_etiquetas.columns = ['Característica (NLP)', 'Cantidad de Anuncios']
conteo_etiquetas['Porcentaje (%)'] = (conteo_etiquetas['Cantidad de Anuncios'] / len(muestra_nlp) * 100).round(1)

print("\n=== REPORTE DE EXTRACCIÓN NLP (MUESTRA N=100) ===")
display(conteo_etiquetas)

Archivo de resultados guardado exitosamente en: ../datalake_procesado/nlp/chilepropiedades_nlp_enriquecido.jsonl

=== REPORTE DE EXTRACCIÓN NLP (MUESTRA N=100) ===


,Característica (NLP),Cantidad de Anuncios,Porcentaje (%)
0,buena conectividad,92,92.0
1,ampliación o remodelada,91,91.0
2,cercano al metro,71,71.0
3,terraza o patio,70,70.0
4,piscina o quincho,33,33.0
5,estacionamiento,25,25.0
6,bodega,20,20.0
7,seguridad o condominio cerrado,17,17.0
